In [ ]:
# ==== Cell 0: Project setup and reproducibility ====
from pathlib import Path
import os, random, numpy as np, torch

# ---------- Directory setup ----------
PROJECT_ROOT = Path.cwd()
DATASET_ROOT = PROJECT_ROOT / "dataset"
PAIRS_CSV    = PROJECT_ROOT / "image_label_pairs.csv"   # will be created later
LOGS_DIR     = PROJECT_ROOT / "logs"
CKPT_DIR     = PROJECT_ROOT / "checkpoints"
SAMPLES_DIR  = PROJECT_ROOT / "samples"

for d in [LOGS_DIR, CKPT_DIR, SAMPLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------- Dataset subfolders ----------
# Add or remove camera folders here as needed
CAMERA_PATHS = [
    "m2020/ncam",
    "m2020/mcam",
]

# ---------- Reproducibility ----------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ---------- Summary ----------
print("Project structure initialized:")
print(f"  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"  DATASET_ROOT : {DATASET_ROOT}")
print(f"  LOGS_DIR     : {LOGS_DIR}")
print(f"  CKPT_DIR     : {CKPT_DIR}")
print(f"  SAMPLES_DIR  : {SAMPLES_DIR}")
print(f"  CAMERA_PATHS : {CAMERA_PATHS}")


In [ ]:
# ==== Cell 1: Scan cameras, report counts, and build image_label_pairs.csv ====
from pathlib import Path
from PIL import Image
import csv, re
import numpy as np

# Uses: PROJECT_ROOT, DATASET_ROOT, CAMERA_PATHS, PAIRS_CSV from Cell 0

# Normalize common suffixes in filenames (e.g., *_merged, *_mask)
_SUFFIX_RE = re.compile(r'(?:_merged\d*|_mask\d*)$', flags=re.IGNORECASE)
def norm_stem(name: str) -> str:
    return _SUFFIX_RE.sub('', name)

def folder_summary_and_pairs(cam_rel_path: str):
    img_dir = DATASET_ROOT / cam_rel_path / "images"
    lbl_dir = DATASET_ROOT / cam_rel_path / "labels"

    if not img_dir.exists() or not lbl_dir.exists():
        print(f"[{cam_rel_path}] Missing required folders: {img_dir} or {lbl_dir}")
        return [], 0, 0, 0, (np.nan, np.nan)

    img_files = sorted([p for p in img_dir.glob("*.*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}])
    lbl_files = sorted(lbl_dir.glob("*.png"))

    # Build label lookup by exact stem and normalized stem
    label_by_stem = {}
    for lp in lbl_files:
        if lp.stem not in label_by_stem:
            label_by_stem[lp.stem] = lp
    label_by_norm = {}
    for lp in lbl_files:
        n = norm_stem(lp.stem)
        label_by_norm.setdefault(n, []).append(lp)

    # Pairing driven by image filenames
    pairs = []
    sizes_w, sizes_h = [], []
    missing = 0
    for ip in img_files:
        try:
            with Image.open(ip) as im:
                w, h = im.size
                sizes_w.append(w); sizes_h.append(h)
        except Exception:
            pass

        stem = ip.stem
        lbl = label_by_stem.get(stem)
        if lbl is None:
            n = norm_stem(stem)
            cands = label_by_norm.get(n, [])
            if len(cands) == 1:
                lbl = cands[0]
            elif len(cands) > 1:
                # Prefer a candidate whose stem starts with the image stem
                exact_like = [c for c in cands if c.stem.startswith(stem)]
                lbl = exact_like[0] if exact_like else cands[0]

        if lbl is not None:
            pairs.append((
                str((ip).relative_to(PROJECT_ROOT)),
                str((lbl).relative_to(PROJECT_ROOT))
            ))
        else:
            missing += 1

    mean_w = float(np.mean(sizes_w)) if sizes_w else float('nan')
    mean_h = float(np.mean(sizes_h)) if sizes_h else float('nan')

    print(f"[{cam_rel_path}] images: {len(img_files)} | labels: {len(lbl_files)} | matched pairs: {len(pairs)} | images without labels: {missing}")
    print(f"[{cam_rel_path}] mean resolution: {mean_w:.1f} x {mean_h:.1f}")

    return pairs, len(img_files), len(lbl_files), len(pairs), (mean_w, mean_h)

# Build pairs for all camera paths
all_pairs = []
tot_imgs = tot_lbls = tot_pairs = 0
for cam_path in CAMERA_PATHS:
    pairs, ni, nl, npairs, _ = folder_summary_and_pairs(cam_path)
    all_pairs.extend(pairs)
    tot_imgs  += ni
    tot_lbls  += nl
    tot_pairs += npairs

# Save CSV at project root
with open(PAIRS_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_path", "label_path"])
    writer.writerows(all_pairs)

print("\nOverall summary")
print(f"  total images: {tot_imgs}")
print(f"  total labels: {tot_lbls}")
print(f"  total matched pairs: {tot_pairs}")
print(f"Saved pairs CSV: {PAIRS_CSV}")


In [ ]:
# ==== Cell 2: Label remapping utilities (GEO → 4-class NAV) ====
import numpy as np

VALID_NAV_SET = {0, 1, 2, 3, 255}

def _ensure_single_channel(mask_np: np.ndarray) -> np.ndarray:
    """If mask is RGB/paletted, collapse to single channel."""
    if mask_np.ndim == 3 and mask_np.shape[-1] >= 1:
        mask_np = mask_np[..., 0]
    return mask_np.astype(np.uint8, copy=False)

def remap_labels(mask_np: np.ndarray) -> np.ndarray:
    """
    Returns a mask with labels in {0:soil, 1:bedrock, 2:sand, 3:big rock, 255:ignore}.
    If the mask is already in {0,1,2,3,255}, it is returned unchanged.
    GEO labels are mapped as:
      bedrock family (0–6)     → 1
      float rock family (10–17)→ 3
      sand family (20–22)      → 2
      pebbles/hill (30,50)     → 0
      veins (40)               → 3
      255                      → 255
      anything else            → 255
    """
    mask_np = _ensure_single_channel(mask_np)

    uniq = set(np.unique(mask_np).tolist())
    if uniq.issubset(VALID_NAV_SET):
        return mask_np  # already NAV-style

    remapped = np.full_like(mask_np, fill_value=255, dtype=np.uint8)

    # soil
    soil_ids = {30, 50}
    remapped[np.isin(mask_np, list(soil_ids))] = 0

    # bedrock family
    bedrock_ids = list(range(0, 7))
    remapped[np.isin(mask_np, bedrock_ids)] = 1

    # sand family
    sand_ids = [20, 21, 22]
    remapped[np.isin(mask_np, sand_ids)] = 2

    # big rock / float rocks / veins
    bigrock_ids = list(range(10, 18)) + [40]
    remapped[np.isin(mask_np, bigrock_ids)] = 3

    # keep explicit ignores
    remapped[mask_np == 255] = 255
    return remapped


In [ ]:
# ==== Cell 3: Dataset and Dataloaders ====
import csv, random
from pathlib import Path
from typing import List, Tuple
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

# Configuration
CROP_SIZE = 512
BATCH_SIZE = 8
NUM_WORKERS = 4
IGNORE_INDEX = 255
USE_IMAGENET_NORM = True
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def _load_pairs(csv_path: Path) -> List[Tuple[Path, Path]]:
    pairs = []
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_p = PROJECT_ROOT / row["image_path"]
            lbl_p = PROJECT_ROOT / row["label_path"]
            if img_p.exists() and lbl_p.exists():
                pairs.append((img_p, lbl_p))
    return pairs

def _resize_pad_crop(img: Image.Image, mask: Image.Image, size: int = CROP_SIZE):
    w, h = img.size
    scale = max(1.0, size / min(w, h))
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    img  = img.resize((new_w, new_h), Image.BILINEAR)
    mask = mask.resize((new_w, new_h), Image.NEAREST)
    pad_w, pad_h = max(0, size - new_w), max(0, size - new_h)
    if pad_w or pad_h:
        img  = TF.pad(img, [0, 0, pad_w, pad_h], fill=0)
        mask = TF.pad(mask, [0, 0, pad_w, pad_h], fill=IGNORE_INDEX)
    img  = TF.center_crop(img, [size, size])
    mask = TF.center_crop(mask, [size, size])
    return img, mask

def _to_tensors(img: Image.Image, mask: Image.Image):
    img_t = TF.to_tensor(img)
    if USE_IMAGENET_NORM:
        img_t = TF.normalize(img_t, mean=IMAGENET_MEAN, std=IMAGENET_STD)
    mask_t = torch.from_numpy(np.array(mask, dtype=np.uint8)).long()
    return img_t, mask_t

class MarsDataset(Dataset):
    def __init__(self, pairs: List[Tuple[Path, Path]], split: str = "train"):
        self.pairs = pairs
        self.split = split

    def __len__(self): return len(self.pairs)

    def __getitem__(self, i):
        img_p, lbl_p = self.pairs[i]
        img = Image.open(img_p).convert("RGB")
        mask_np = np.array(Image.open(lbl_p), dtype=np.uint8)
        mask_np = remap_labels(mask_np)  # from Cell 2
        mask = Image.fromarray(mask_np)

        # simple augmentations
        if self.split == "train":
            if random.random() < 0.5:
                img, mask = TF.hflip(img), TF.hflip(mask)
            angle = random.uniform(-10, 10)
            img  = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)
        img, mask = _resize_pad_crop(img, mask, CROP_SIZE)
        img_t, mask_t = _to_tensors(img, mask)
        return img_t, mask_t

# Build pairs and dataloaders
all_pairs = _load_pairs(PAIRS_CSV)
n = len(all_pairs)
train_split, val_split = int(0.7 * n), int(0.9 * n)
train_pairs = all_pairs[:train_split]
val_pairs   = all_pairs[train_split:val_split]
test_pairs  = all_pairs[val_split:]

train_ds = MarsDataset(train_pairs, split="train")
val_ds   = MarsDataset(val_pairs, split="val")
test_ds  = MarsDataset(test_pairs, split="test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Total pairs: {n} | Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


In [ ]:
# ==== Cell 4: Hyperparameters + import/instantiate model ====
import importlib, json, torch
from datetime import datetime
from pathlib import Path

# ------- Hyperparameters (training + loss) -------
EPOCHS        = 60
BASE_LR       = 1e-2
MOMENTUM      = 0.9
WEIGHT_DECAY  = 1e-4
POLY_POWER    = 0.9          # for polynomial LR decay
ALPHA_BCE     = 0.5          # loss mix: BCE weight
BETA_IOU      = 0.5          # loss mix: soft IoU weight

# ------- Model settings -------
MODEL_MODULE  = "segmarsvit_model"  # swap this to switch architectures
IN_CHANNELS   = 3
NUM_CLASSES   = 4                   # 0: soil, 1: bedrock, 2: sand, 3: big rock

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_NAME = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ------- Import and build the model -------
mmod = importlib.import_module(MODEL_MODULE)
build_segmarsvit = getattr(mmod, "build_segmarsvit")
count_parameters = getattr(mmod, "count_parameters")

model = build_segmarsvit(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES, device=DEVICE)
print(f"Model: {MODEL_MODULE}.{model.__class__.__name__}")
print("Device:", DEVICE)
print("Trainable parameters:", count_parameters(model))

# ------- Save a tiny run config for traceability -------
run_cfg = {
    "run_name": RUN_NAME,
    "device": str(DEVICE),
    "epochs": EPOCHS,
    "base_lr": BASE_LR,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "poly_power": POLY_POWER,
    "alpha_bce": ALPHA_BCE,
    "beta_iou": BETA_IOU,
    "model_module": MODEL_MODULE,
    "in_channels": IN_CHANNELS,
    "num_classes": NUM_CLASSES,
}
(Path(LOGS_DIR) / f"{RUN_NAME}_config.json").write_text(json.dumps(run_cfg, indent=2))
print("Saved run config to:", Path(LOGS_DIR) / f"{RUN_NAME}_config.json")


In [ ]:
# ==== Cell 5: Losses, metrics, and evaluation ====
import torch
import torch.nn.functional as F

IGNORE_INDEX = 255  # same as dataset
NUM_CLASSES = 4

def one_hot_targets(y, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX):
    mask_valid = (y != ignore_index)
    y_valid = y.clone()
    y_valid[~mask_valid] = 0
    oh = F.one_hot(y_valid, num_classes=num_classes).permute(0, 3, 1, 2).float()
    oh = oh * mask_valid.unsqueeze(1)
    return oh, mask_valid

@torch.no_grad()
def class_frequency(oh):
    return oh.sum(dim=(0, 2, 3))  # (C,)

def weighted_bce_with_logits(logits, oh, valid_mask, num_classes=NUM_CLASSES, eps=1e-6):
    with torch.no_grad():
        counts = class_frequency(oh) + eps
        inv = 1.0 / counts
        w = (inv / inv.sum() * num_classes).to(logits.device)
        w = w.view(1, -1, 1, 1)
    bce = F.binary_cross_entropy_with_logits(logits, oh, weight=w, reduction='none')
    bce = bce * valid_mask.unsqueeze(1)
    denom = valid_mask.sum().clamp_min(1).float()
    return bce.sum() / denom

def soft_iou_loss(logits, oh, valid_mask, num_classes=NUM_CLASSES, eps=1e-6):
    prob = torch.softmax(logits, dim=1)
    prob = prob * valid_mask.unsqueeze(1)
    oh   = oh   * valid_mask.unsqueeze(1)
    inter = (prob * oh).sum(dim=(0,2,3))
    union = (prob + oh - prob * oh).sum(dim=(0,2,3)) + eps
    with torch.no_grad():
        counts = class_frequency(oh) + eps
        inv = 1.0 / counts
        w = (inv / inv.sum() * num_classes).to(logits.device)
    iou = inter / union
    miou_w = (w * iou).sum() / w.sum().clamp_min(1e-6)
    return 1.0 - miou_w

@torch.no_grad()
def evaluate(model, loader, device, alpha_bce=0.5, beta_iou=0.5):
    model.eval()
    conf = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=device)
    val_loss = 0.0
    n_batches = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        oh, valid = one_hot_targets(targets)
        loss = alpha_bce * weighted_bce_with_logits(logits, oh, valid) + \
               beta_iou * soft_iou_loss(logits, oh, valid)
        val_loss += loss.item()
        n_batches += 1

        preds = torch.argmax(logits, dim=1)
        valid_idx = valid.view(-1)
        t = targets.view(-1)[valid_idx]
        p = preds.view(-1)[valid_idx]
        k = (t >= 0) & (t < NUM_CLASSES)
        t, p = t[k], p[k]
        inds = NUM_CLASSES * t + p
        binc = torch.bincount(inds, minlength=NUM_CLASSES*NUM_CLASSES)
        conf += binc.reshape(NUM_CLASSES, NUM_CLASSES)

    tp = conf.diag().float()
    pos = conf.sum(dim=1).float()
    pred_pos = conf.sum(dim=0).float()
    union = pos + pred_pos - tp
    iou = torch.where(union > 0, tp / union, torch.zeros_like(tp))
    miou = iou[iou == iou].mean().item() if (union > 0).any() else 0.0
    pix_acc = (tp.sum() / conf.sum().clamp_min(1)).item()
    return val_loss / max(1, n_batches), miou, pix_acc, iou.detach().cpu().tolist()

print("✅ Losses and evaluation helpers ready.")


In [ ]:
# ==== Cell 6: Training loop with periodic checkpoint saving ====
import time, csv, torch
from torch.optim import SGD
from torch.optim.lr_scheduler import LambdaLR

# --- Configuration (hyperparams from Cell 4) ---
EPOCHS        = EPOCHS
BASE_LR       = BASE_LR
MOMENTUM      = MOMENTUM
WEIGHT_DECAY  = WEIGHT_DECAY
POLY_POWER    = POLY_POWER
ALPHA_BCE     = ALPHA_BCE
BETA_IOU      = BETA_IOU

# --- Initialize optimizer and scheduler ---
optimizer = SGD(model.parameters(), lr=BASE_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
total_iters = EPOCHS * max(1, len(train_loader))
lr_lambda = lambda it: (1 - min(it, total_iters) / float(total_iters)) ** POLY_POWER
scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

# --- Logging ---
log_csv = LOGS_DIR / f"{RUN_NAME}_train_log.csv"
with open(log_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_mIoU", "val_pixAcc", "lr"])

best_miou = -1.0
global_iter = 0

# --- Training loop ---
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    start_time = time.time()

    for images, targets in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        logits = model(images)
        oh, valid = one_hot_targets(targets)
        loss = ALPHA_BCE * weighted_bce_with_logits(logits, oh, valid) + \
               BETA_IOU * soft_iou_loss(logits, oh, valid)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        n_batches += 1
        global_iter += 1

    train_loss = epoch_loss / max(1, n_batches)
    val_loss, val_miou, val_pixacc, class_iou = evaluate(model, val_loader, DEVICE, ALPHA_BCE, BETA_IOU)

    # --- Save best checkpoint ---
    if val_miou > best_miou:
        best_miou = val_miou
        best_path = CKPT_DIR / f"{RUN_NAME}_segmarsvit_best.pth"
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_miou": best_miou,
        }, best_path)

    # --- Save periodic checkpoint (every 5 epochs) ---
    if epoch % 5 == 0 or epoch == EPOCHS:
        last_path = CKPT_DIR / f"{RUN_NAME}_segmarsvit_last.pth"
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_miou": best_miou,
        }, last_path)

    # --- Logging ---
    with open(log_csv, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch,
            f"{train_loss:.6f}",
            f"{val_loss:.6f}",
            f"{val_miou:.4f}",
            f"{val_pixacc:.4f}",
            f"{optimizer.param_groups[0]['lr']:.6e}",
        ])

    elapsed = time.time() - start_time
    print(f"Epoch {epoch:03d}/{EPOCHS} | Train {train_loss:.4f} | Val {val_loss:.4f} | "
          f"mIoU {val_miou:.4f} | PixAcc {val_pixacc:.4f} | Time {elapsed/60:.1f}m")

print(f"\nTraining complete. Best validation mIoU: {best_miou:.4f}")
print(f"Checkpoints saved to {CKPT_DIR}")


In [ ]:
# ==== Cell 7: Final test evaluation + side-by-side visuals (image | ground-truth | prediction) ====
from pathlib import Path
import json, numpy as np, torch
from PIL import Image
import torchvision.transforms.functional as TF
from tqdm import tqdm

# --- Paths
best_ckpt = CKPT_DIR / f"{RUN_NAME}_segmarsvit_best.pth"
last_ckpt = CKPT_DIR / f"{RUN_NAME}_segmarsvit_last.pth"
out_dir   = SAMPLES_DIR / f"{RUN_NAME}_test_comparisons"
out_dir.mkdir(parents=True, exist_ok=True)

# --- Load checkpoint (prefer best, else last)
ckpt_path = best_ckpt if best_ckpt.exists() else last_ckpt
assert ckpt_path.exists(), f"No checkpoint found at {best_ckpt} or {last_ckpt}"
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# --- Color palette: 0 soil, 1 bedrock, 2 sand, 3 big rock
PALETTE = {
    0: (110, 65, 40),   # soil (brown)
    1: (160, 160, 160), # bedrock (gray)
    2: (230, 200, 70),  # sand (yellow)
    3: (70, 110, 200),  # big rock (blue)
}
IGNORE_INDEX = 255
NUM_CLASSES = 4

def colorize_mask(mask_np: np.ndarray) -> Image.Image:
    h, w = mask_np.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for k, c in PALETTE.items():
        rgb[mask_np == k] = c
    # visualize ignore as black (stay 0,0,0)
    return Image.fromarray(rgb, mode="RGB")

def denorm_image(t: torch.Tensor) -> torch.Tensor:
    # t: [3,H,W], undo ImageNet norm if it was applied
    use_norm = globals().get("USE_IMAGENET_NORM", True)
    if use_norm:
        mean = torch.tensor([0.485, 0.456, 0.406], device=t.device).view(3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=t.device).view(3,1,1)
        t = t * std + mean
    return t.clamp(0,1)

# --- Metrics containers
conf = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=DEVICE)
saved_count = 0

# --- Test loop
with torch.no_grad():
    for idx, (images, targets) in enumerate(tqdm(test_loader, desc="Testing")):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        logits = model(images)
        preds = torch.argmax(logits, dim=1)  # [B,H,W]

        # update metrics (valid pixels only)
        valid = (targets != IGNORE_INDEX)
        t = targets.view(-1)[valid.view(-1)]
        p = preds.view(-1)[valid.view(-1)]
        k = (t >= 0) & (t < NUM_CLASSES)
        t = t[k]; p = p[k]
        inds = NUM_CLASSES * t + p
        binc = torch.bincount(inds, minlength=NUM_CLASSES*NUM_CLASSES)
        conf += binc.reshape(NUM_CLASSES, NUM_CLASSES)

        # save side-by-side composite for each item in batch
        for b in range(images.size(0)):
            img_vis = denorm_image(images[b].cpu())
            img_pil = TF.to_pil_image(img_vis)

            gt_np   = targets[b].cpu().numpy().astype(np.uint8)
            pred_np = preds[b].cpu().numpy().astype(np.uint8)

            gt_rgb   = colorize_mask(np.where(gt_np==IGNORE_INDEX, 0, gt_np))
            pred_rgb = colorize_mask(pred_np)

            # [Original | Ground Truth | Prediction]
            w, h = img_pil.size
            canvas = Image.new("RGB", (w * 3, h))
            canvas.paste(img_pil, (0, 0))
            canvas.paste(gt_rgb,  (w, 0))
            canvas.paste(pred_rgb,(w * 2, 0))

            save_path = out_dir / f"cmp_{idx:04d}_{b}.png"
            canvas.save(save_path)
            saved_count += 1

# --- Final test metrics
tp = conf.diag().float()
pos = conf.sum(dim=1).float()
pred_pos = conf.sum(dim=0).float()
union = pos + pred_pos - tp
iou = torch.where(union > 0, tp / union, torch.zeros_like(tp))
miou = iou[iou == iou].mean().item() if (union > 0).any() else 0.0
pix_acc = (tp.sum() / conf.sum().clamp_min(1)).item()

metrics = {
    "checkpoint": str(ckpt_path),
    "test_mIoU": round(miou, 4),
    "test_pixel_accuracy": round(pix_acc, 4),
    "per_class_IoU": [round(v, 4) for v in iou.detach().cpu().tolist()],
    "comparisons_saved": saved_count,
}
(Path(LOGS_DIR) / f"{RUN_NAME}_test_metrics.json").write_text(json.dumps(metrics, indent=2))

print("Test metrics:", metrics)
print("Saved side-by-side comparisons to:", out_dir)
